# A1 -- Milling Machine Failure Classification

**Asset**: AI4I 2020 Predictive Maintenance Dataset (10,000 machined parts)  
**ML Paradigm**: Supervised binary classification + multi-label per-failure-mode analysis  
**Objective**: Predict machine failure from process signals (temperature, speed, torque, tool wear)  
**Ground-Truth Policy**: `Status/*` labels are targets only -- never used as input features  

---

## 1. Environment Setup and Data Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score,
    average_precision_score
)
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
import joblib
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)

REPO_ROOT = Path("..").resolve()
DATASET_PATH = REPO_ROOT / "ai4i+2020+predictive+maintenance+dataset" / "ai4i2020.csv"
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

print(f"Dataset: {DATASET_PATH}")
print(f"Model output: {MODEL_DIR.resolve()}")

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Data Understanding and Preprocessing

### 2.1 Feature Dictionary

| Feature | Unit | Role |
|---|---|---|
| Air temperature [K] | Kelvin | Process input |
| Process temperature [K] | Kelvin | Process input |
| Rotational speed [rpm] | RPM | Drive parameter |
| Torque [Nm] | Newton-metre | Drive parameter |
| Tool wear [min] | Minutes | Cumulative wear |
| Type | Categorical (L/M/H) | Product quality class |
| Machine failure | Binary | Target (aggregate) |
| TWF, HDF, PWF, OSF, RNF | Binary | Target (per-mode) |

In [ ]:
# Rename columns for cleaner code
df = df.rename(columns={
    "Air temperature [K]": "AirTemp_K",
    "Process temperature [K]": "ProcTemp_K",
    "Rotational speed [rpm]": "Speed_rpm",
    "Torque [Nm]": "Torque_Nm",
    "Tool wear [min]": "ToolWear_min",
    "Machine failure": "MachineFailure",
})

# Derived feature: product power (as computed by the PLC)
df["Power_W"] = df["Torque_Nm"] * df["Speed_rpm"] * 2 * np.pi / 60

# Derived feature: temperature delta (key for HDF detection)
df["DeltaT_K"] = df["ProcTemp_K"] - df["AirTemp_K"]

# Derived feature: overstrain product (key for OSF detection)
df["WearTorque"] = df["ToolWear_min"] * df["Torque_Nm"]

print("Derived features added: Power_W, DeltaT_K, WearTorque")
print(f"Dataset shape after feature engineering: {df.shape}")

In [ ]:
# Encode product type ordinally: L=0, M=1, H=2
type_encoder = OrdinalEncoder(categories=[["L", "M", "H"]])
df["Type_enc"] = type_encoder.fit_transform(df[["Type"]]).astype(int)

# Statistical summary of process features
process_features = [
    "AirTemp_K", "ProcTemp_K", "Speed_rpm", "Torque_Nm",
    "ToolWear_min", "Power_W", "DeltaT_K", "WearTorque", "Type_enc"
]
df[process_features].describe().round(2)

## 3. Class Imbalance Analysis

In [ ]:
failure_modes = ["MachineFailure", "TWF", "HDF", "PWF", "OSF", "RNF"]

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
for ax, mode in zip(axes, failure_modes):
    counts = df[mode].value_counts()
    pct = counts[1] / len(df) * 100 if 1 in counts.index else 0
    ax.bar(
        ["Normal", "Failure"],
        [counts.get(0, 0), counts.get(1, 0)],
        color=["#2ecc71", "#e74c3c"],
    )
    ax.set_title(f"{mode}\n({pct:.2f}%)")
    ax.set_ylabel("Count")
plt.suptitle(
    "Failure Mode Distribution (Severe Class Imbalance)",
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.savefig("models/a1_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nFailure counts:")
for mode in failure_modes:
    n = df[mode].sum()
    print(f"  {mode:20s}: {n:5d} / {len(df)} ({n/len(df)*100:.2f}%)")

## 4. Feature Distributions by Failure Status

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, feat in zip(axes.flat, process_features[:-1]):
    for label, color, name in [(0, "#2ecc71", "Normal"), (1, "#e74c3c", "Failure")]:
        subset = df[df["MachineFailure"] == label][feat]
        ax.hist(subset, bins=50, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

ax_last = axes.flat[-1]
ax_last.axis("off")
ax_last.text(
    0.5, 0.5, "8 input features\n+ 1 encoded type",
    ha="center", va="center", fontsize=12, style="italic",
)

plt.suptitle("Feature Distributions: Normal vs Failure", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("models/a1_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Correlation matrix of process features
corr = df[process_features + ["MachineFailure"]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, square=True, ax=ax, linewidths=0.5,
)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("models/a1_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Train/Test Split

Time-ordered 80/20 split as specified in the roadmap. No random shuffle to prevent data leakage at the dataset wrap boundary.

In [ ]:
FEATURE_COLS = process_features  # 9 features
TARGET_COL = "MachineFailure"
FAILURE_MODES = ["TWF", "HDF", "PWF", "OSF", "RNF"]

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values
y_modes = df[FAILURE_MODES].values

from sklearn.model_selection import train_test_split

# Stratified random split to ensure all failure modes appear in both train and test
X_train, X_test, y_train, y_test, y_modes_train, y_modes_test = train_test_split(
    X, y, y_modes, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples "
      f"({y_train.sum()} failures, {y_train.mean()*100:.2f}%)")
print(f"Test set:     {X_test.shape[0]} samples "
      f"({y_test.sum()} failures, {y_test.mean()*100:.2f}%)")

# Feature scaling (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Binary Classification -- Machine Failure

XGBoost with `scale_pos_weight` to handle the 96.6% / 3.4% class imbalance.

In [ ]:
# Compute scale_pos_weight for imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

# XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="logloss",
    verbosity=0,
)

# Cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv, scoring="f1")
cv_auc = cross_val_score(
    xgb_model, X_train_scaled, y_train, cv=cv, scoring="roc_auc"
)

print(f"5-Fold CV F1:      {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
print(f"5-Fold CV ROC-AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})")

# Final fit on full training set
xgb_model.fit(X_train_scaled, y_train)
print("\nModel trained on full training set.")

## 7. Binary Classification Evaluation

In [ ]:
y_pred = xgb_model.predict(X_test_scaled)
y_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("Classification Report (Binary: Machine Failure)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Normal", "Failure"]))

roc_auc = roc_auc_score(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
print(f"ROC-AUC:            {roc_auc:.4f}")
print(f"Average Precision:  {avg_prec:.4f}")
print(f"F1-Score:           {f1:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Failure"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Confusion Matrix")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="#3498db", lw=2, label=f"ROC-AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(rec, prec, color="#e74c3c", lw=2, label=f"AP = {avg_prec:.3f}")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall Curve")
axes[2].legend()

plt.suptitle("Binary Classification Performance", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("models/a1_binary_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# XGBoost native feature importance (gain-based)
importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": importances,
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_imp["Feature"], feat_imp["Importance"], color="#3498db")
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("XGBoost Feature Importance -- Binary Failure Classifier")
plt.tight_layout()
plt.savefig("models/a1_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# SHAP analysis for model interpretability
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_scaled)

fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test_scaled, feature_names=FEATURE_COLS, show=False,
)
plt.title("SHAP Feature Impact on Machine Failure Prediction")
plt.tight_layout()
plt.savefig("models/a1_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Multi-Label Analysis -- Per Failure Mode

Train separate binary classifiers for each of the 5 failure modes:
TWF (Tool Wear), HDF (Heat Dissipation), PWF (Power), OSF (Overstrain), RNF (Random).

In [ ]:
mode_models = {}
mode_results = {}

for i, mode in enumerate(FAILURE_MODES):
    y_m_train = y_modes_train[:, i]
    y_m_test = y_modes_test[:, i]

    if y_m_train.sum() == 0:
        print(f"{mode}: No positive samples in training set, skipping.")
        continue

    neg = (y_m_train == 0).sum()
    pos = (y_m_train == 1).sum()
    
    # Apply SMOTE to handle extreme imbalance, but only up to 20% of the majority class
    # to avoid creating too much synthetic noise.
    from imblearn.over_sampling import SMOTE
    smote_ratio = min(1.0, (neg * 0.2) / max(pos, 1))
    
    if pos < 5:
        # If extremely few positives (e.g. 1 or 2), SMOTE will fail. Fall back to simple XGBoost with high weight.
        X_res, y_m_res = X_train_scaled, y_m_train
    else:
        try:
            smote = SMOTE(sampling_strategy=smote_ratio, k_neighbors=min(pos-1, 5), random_state=42)
            X_res, y_m_res = smote.fit_resample(X_train_scaled, y_m_train)
        except ValueError:
            X_res, y_m_res = X_train_scaled, y_m_train
            
    # Recompute scale_pos_weight after SMOTE
    neg_res = (y_m_res == 0).sum()
    pos_res = (y_m_res == 1).sum()
    spw = neg_res / max(pos_res, 1)

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=spw,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
    )
    model.fit(X_res, y_m_res)

    # Find optimal threshold to maximize F1 on the training set
    y_train_prob = model.predict_proba(X_res)[:, 1]
    best_t, best_f1_train = 0.5, 0.0
    for t in np.arange(0.1, 0.95, 0.05):
        f1_t = f1_score(y_m_res, (y_train_prob > t).astype(int), zero_division=0)
        if f1_t > best_f1_train:
            best_f1_train, best_t = f1_t, t

    y_m_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Apply optimal threshold
    y_m_pred = (y_m_prob >= best_t).astype(int)

    f1_val = f1_score(y_m_test, y_m_pred, zero_division=0)
    recall_val = float(
        y_m_pred[y_m_test == 1].sum() / max(y_m_test.sum(), 1)
    )
    precision_val = float(
        y_m_test[y_m_pred == 1].sum() / max(y_m_pred.sum(), 1)
    ) if y_m_pred.sum() > 0 else 0.0

    mode_models[mode] = {"model": model, "threshold": best_t}
    mode_results[mode] = {
        "train_pos": int(pos),
        "test_pos": int(y_m_test.sum()),
        "f1": f1_val,
        "recall": recall_val,
        "precision": precision_val,
        "threshold": best_t,
    }
    print(
        f"{mode:4s} | train_pos={pos:3d} test_pos={int(y_m_test.sum()):3d} | "
        f"F1={f1_val:.3f} Recall={recall_val:.3f} Precision={precision_val:.3f} (t={best_t:.2f})"
    )

print("\nPer-mode classifiers trained.")

In [ ]:
# Visualize per-mode performance
if mode_results:
    results_df = pd.DataFrame(mode_results).T
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(results_df))
    w = 0.25
    ax.bar(x - w, results_df["f1"], w, label="F1", color="#3498db")
    ax.bar(x, results_df["recall"], w, label="Recall", color="#e74c3c")
    ax.bar(x + w, results_df["precision"], w, label="Precision", color="#2ecc71")
    ax.set_xticks(x)
    ax.set_xticklabels(results_df.index)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Per-Failure-Mode Classification Performance")
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        "models/a1_per_mode_performance.png", dpi=150, bbox_inches="tight"
    )
    plt.show()

## 10. Model Export

In [ ]:
# Save binary classifier
joblib.dump(
    {
        "model": xgb_model,
        "scaler": scaler,
        "type_encoder": type_encoder,
        "feature_columns": FEATURE_COLS,
        "target": TARGET_COL,
    },
    MODEL_DIR / "a1_classifier.joblib",
)

# Save per-mode classifiers
for mode, model in mode_models.items():
    joblib.dump(model, MODEL_DIR / f"a1_{mode.lower()}_classifier.joblib")

print(f"Models saved to {MODEL_DIR.resolve()}")
for f in sorted(MODEL_DIR.glob("a1_*.joblib")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 11. Verification Summary

In [ ]:
print("=" * 60)
print("A1 FAILURE CLASSIFICATION -- VERIFICATION SUMMARY")
print("=" * 60)
print()

print(f"Binary Classifier (Machine Failure):")
print(f"  ROC-AUC:          {roc_auc:.4f}  "
      f"{'PASS' if roc_auc >= 0.85 else 'BELOW TARGET'} (target >= 0.85)")
print(f"  F1-Score:         {f1:.4f}  "
      f"{'PASS' if f1 >= 0.50 else 'BELOW TARGET'} (target >= 0.50)")
print()

print("Per-Mode Classifiers:")
for mode, res in mode_results.items():
    status = "PASS" if res["recall"] > 0 else "CHECK"
    print(f"  {mode:4s}: F1={res['f1']:.3f}  Recall={res['recall']:.3f}  [{status}]")
print()

model_files = list(MODEL_DIR.glob("a1_*.joblib"))
print(f"Model artifacts: {len(model_files)} files saved")
for f in sorted(model_files):
    print(f"  {f.name}")
print()
print("Notebook execution: COMPLETE")